<a href="https://colab.research.google.com/github/Evans-Sense/pet/blob/main/Hometask_6_MLPro2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_csv('/kaggle/input/understanding_cloud_organization/train.csv')

# Separate Image_Label into ImageId and Label
train_df[['ImageId', 'Label']] = train_df['Image_Label'].str.split('_', expand=True)
train_df = train_df.drop(columns=['Image_Label'])

# Fill missing EncodedPixels with NaN
train_df['EncodedPixels'] = train_df['EncodedPixels'].fillna('')

# Pivot the dataframe to have one row per image
train_df = train_df.pivot(index='ImageId', columns='Label', values='EncodedPixels').reset_index()

# Fill missing values with empty strings
train_df.fillna('', inplace=True)

print(train_df.head())

In [ ]:
# Initialize the 'fold' column to -1
train_df['fold'] = -1

# Initialize KFold
kf = KFold(n_splits=9, shuffle=True, random_state=42)

# Assign fold numbers
for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    train_df.loc[val_idx, 'fold'] = fold

# Now select one fold for validation (e.g., fold 0)
train_data = train_df[train_df['fold'] != 0].reset_index(drop=True)
val_data = train_df[train_df['fold'] == 0].reset_index(drop=True)

# Verify the split
print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")

In [5]:
path = kagglehub.dataset_download("competitions/understanding_cloud_organization/data")

print("Path to dataset files:", path)

ValueError: Invalid dataset handle: competitions/understanding_cloud_organization/data

In [ ]:
for dirname, _, filenames in os.walk(path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
df= pd.read_csv('/root/.cache/kagglehub/datasets/uciml/german-credit/versions/1/german_credit_data.csv')

df.head()

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.down1 = DoubleConv(3, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)
        self.up_transpose = nn.ConvTranspose2d(512, 256, 2, stride=2)

        self.up1 = DoubleConv(512, 256)
        self.up2 = DoubleConv(256, 128)
        self.up3 = DoubleConv(128, 64)

        self.out = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool(x1))
        x3 = self.down3(self.pool(x2))
        x4 = self.down4(self.pool(x3))

        x = self.up_transpose(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.up1(x)
        x = self.up2(torch.cat([self.up_transpose(x), x2], dim=1))
        x = self.up3(torch.cat([self.up_transpose(x), x1], dim=1))

        return torch.sigmoid(self.out(x))


In [ ]:
class CloudDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = np.load(self.mask_paths[idx])  # Предполагаем, что маски сохранены как .npy

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask

# Аугментации
transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.Normalize(),
    ToTensorV2(),
])

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return 1 - dice

def dice_coefficient(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    return dice_loss(pred, target).item()

In [ ]:
model = UNet(n_classes=4).cuda()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = dice_loss

# Цикл обучения
for epoch in range(50):
    model.train()
    train_loss = 0
    for images, masks in train_loader:
        images, masks = images.cuda(), masks.cuda()
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Валидация
    model.eval()
    val_dice = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.cuda(), masks.cuda()
            outputs = model(images)
            val_dice += dice_coefficient(outputs, masks)

    print(f'Epoch {epoch}, Train Loss: {train_loss/len(train_loader):.4f}, Val Dice: {val_dice/len(val_loader):.4f}')

In [ ]:
pip install ultralytics

In [ ]:
# Конвертация RLE в YOLO формат
def rle_to_yolo(rle, img_width, img_height):
    mask = rle_to_mask(rle, img_height, img_width)  # Используем функцию из предыдущего решения
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        x_center = (x + w/2) / img_width
        y_center = (y + h/2) / img_height
        w_norm = w / img_width
        h_norm = h / img_height
        boxes.append([class_id, x_center, y_center, w_norm, h_norm])
    return boxes

In [ ]:
from ultralytics import YOLO

# Инициализация модели
model_yolo = YOLO('yolov8n.pt')  # Используем предобученную модель

# Обучение
results = model_yolo.train(
    data='cloud_data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0  # GPU
)

In [ ]:
metrics = model_yolo.val()
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")